In [1]:
import requests
import pandas as pd
import isodate

#### Trouver l'ID de la playlist

In [2]:
# url = "https://youtube.googleapis.com/youtube/v3/channels?part=contentDetails&forHandle=AlexTheAnalyst&key=[]"
# response = requests.get(url)
# res = response.json()
# print(res)

In [3]:
# playlist_id = res["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
# playlist_id

#### Récuperer les IDs des vidéos 

In [4]:
# url1 = 'https://youtube.googleapis.com/youtube/v3/playlistItems?part=contentDetails&playlistId=UU7cs8q-gJRlGwj4A8OmCmXg&key=[]'
# response1 = requests.get(url1)
# response1.json()

## 1. Récupération des identifiants :

#### Fonction pour récupérer l'id de la playlist :

In [5]:
def get_uploads_playlist_id(channel_id, api_key):
    
    url = "https://www.googleapis.com/youtube/v3/channels"
    
    params = {
        "part": "contentDetails",   
        "id": channel_id,          
        "key": api_key              
    }
    
    response = requests.get(url, params=params).json()
    print(response)
    
    playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    
    return playlist_id

#### Fonction pour récupérer les ids des vidéos :

In [6]:
def get_video_ids(playlist_id, api_key):
    
    video_ids = []           
    next_page_token = None   
    
    while True:   
        
        params = {
            "part": "contentDetails",
            "playlistId": playlist_id,  
            "maxResults": 50,           
            "pageToken": next_page_token, 
            "key": api_key
        }
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/playlistItems",
            params=params
        ).json()
        
        for item in response["items"]:
            video_ids.append(item["contentDetails"]["videoId"])
        
        next_page_token = response.get("nextPageToken")
        
        if not next_page_token:
            break   
    
    return video_ids   

#### Fonction pour récupérer les détails des vidéos :

In [7]:
def get_video_details(video_ids, api_key):
    
    all_videos = []   
    
    for i in range(0, len(video_ids), 50):
        
        batch = video_ids[i:i+50]   
        
        params = {
            "part": "snippet,statistics,contentDetails",
            
            "id": ",".join(batch),   
            
            "key": api_key
        }
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/videos",
            params=params
        ).json()
        
        all_videos.extend(response["items"])
    
    return all_videos

## 2. Sauvegarde & transformation des données :

#### Fonction pour sauvegarder les données brutes :

In [8]:
import json
import os

def save_raw(data, filename="videos_raw.json"):

    os.makedirs("data/raw", exist_ok=True)

    filepath = f"data/raw/{filename}"

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

#### Fonction pour sauvegarder les données brutes :

In [9]:
def load_raw(filename="videos_raw.json"):

    filepath = f"data/raw/{filename}"

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    return data

#### Fonction pour extraire les champs et convertir les données brutes en dataframe :

In [10]:
def extract_fields(raw_videos):
    
    records = []   
    
    for v in raw_videos:  
        
        snippet = v.get("snippet", {})
        stats   = v.get("statistics", {})
        content = v.get("contentDetails", {})
        
        records.append({
           "video_id"    : v.get("id"),
            "title"       : snippet.get("title"),
            "published_at": snippet.get("publishedAt"),
            "channel"     : snippet.get("channelTitle"),
            "tags"        : snippet.get("tags", []),
            "duration"    : content.get("duration", "PT0S"),
            "definition"  : content.get("definition"),
            "caption"     : content.get("caption"),
            "views"       : stats.get("viewCount"),
            "likes"       : stats.get("likeCount"),
            "comments"    : stats.get("commentCount"),
            "category_id"  : snippet.get("categoryId"),

        }) 
    df = pd.DataFrame(records)   
    return df 

#### Fonction pour normaliser les champs (convertir dans le bon format):

In [11]:
def normalize(df):

    df["published_at"] = pd.to_datetime(df["published_at"])
    df["year"]         = df["published_at"].dt.year
    df["month"]        = df["published_at"].dt.month


    def convertir_duree(texte):
        try:
            secondes = isodate.parse_duration(texte).total_seconds()
            return int(secondes)
        except:
            return 0  

    df["duration_sec"] = df["duration"].apply(convertir_duree)
    df["duration_min"] = (df["duration_sec"] / 60).round(2)


    for colonne in ["views", "likes", "comments"]:
        df[colonne] = pd.to_numeric(df[colonne], errors="coerce")
        df[colonne] = df[colonne].fillna(0).astype(int)


    df["tags"] = df["tags"].apply(
        lambda x: ",".join(x) if isinstance(x, list) else ""
    )

    df["has_captions"] = df["caption"] == "true"

    return df

#### Fonction pour gérer les valeurs manquantes :

In [12]:
def gerer_manquants(df):
    print(df.isnull().sum())

    df = df.dropna(subset=["title"])

    df["tags"]       = df["tags"].fillna("")
    df["channel"]    = df["channel"].fillna("Inconnu")
    df["definition"] = df["definition"].fillna("Inconnu")
    df["views"]      = df["views"].fillna(0)
    df["likes"]      = df["likes"].fillna(0)
    df["comments"]   = df["comments"].fillna(0)

    return df

#### Fonction principale pour la transformation :

In [13]:
def transform():

    data = load_raw()
    df = extract_fields(data)
    df = normalize(df)

    df = gerer_manquants(df)
    df = df.drop(columns=["duration", "caption"])
    return df

#### Fonction pour sauvegarder le fichier csv final :

In [14]:
def load(df, filename="youtube_data.csv"):

    os.makedirs("data/processed", exist_ok=True)

    filepath = f"data/processed/{filename}"

    df.to_csv(filepath, index=False, encoding="utf-8-sig")

## 3. Pipeline complet :

In [15]:
from dotenv import load_dotenv

load_dotenv()
API_KEY    = os.getenv("YOUTUBE_API_KEY")
CHANNEL_ID = os.getenv("CHANNEL_ID")

playlist_id = get_uploads_playlist_id(CHANNEL_ID, API_KEY)
video_ids = get_video_ids(playlist_id, API_KEY)
raw_data = get_video_details(video_ids, API_KEY)
save_raw(raw_data)

df = transform() 

load(df)

{'kind': 'youtube#channelListResponse', 'etag': 'Gbu6fjt2uOexIf4deIrnb-LY958', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': 'xDiHSFF8exKFo3s4XYEScnJuZpk', 'id': 'UC7cs8q-gJRlGwj4A8OmCmXg', 'contentDetails': {'relatedPlaylists': {'likes': '', 'uploads': 'UU7cs8q-gJRlGwj4A8OmCmXg'}}}]}
video_id        0
title           0
published_at    0
channel         0
tags            0
duration        0
definition      0
caption         0
views           0
likes           0
comments        0
category_id     0
year            0
month           0
duration_sec    0
duration_min    0
has_captions    0
dtype: int64
